In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Get the directory where you are currently working
current_path = os.getcwd()

# Join it with '..' to go up one level to the main folder
parent_path = os.path.abspath(os.path.join(current_path, '..'))

# Add it to sys.path
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

import numpy as np
import scipy as sp
from conversions import cm_to_m, erg_to_joule
from atmospheres.atmosphere_setting import Atmosphere, wind_microphysics
from bulk_param_examination import make_simple_atmosphere, sweep_parameter
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.colors as colors
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from stellar_cradle import solar_mass, get_escape_fluxes
import math







In [ ]:
### Cell with much used parameters are values ###

M_earth = 5.9722 * 10**24               #[kg]        mass of the earth
au = 149597870700 #[m] distance from the sun to the earth
F_xuv_earth = 0.2198576545 #[W/m^2] solar flux at the earth 
solar_mass = 1.989 * 10**30 #[kg] mass of the sun

species_microphysics = {
    #NB, nu_0, mu_wind, mu_plus_wind are for the singly ionised and fully dissociated equivalents!!!!! mu_photo and epsilon xuv is for not dissociated.
    'H2': {'nu_0': 3.288467085473 * 10**15, 'mu_wind': 0.5, 'mu_plus_wind': 1, 'mu_photo': 2, 'epsilon_xuv': wind_microphysics['H2']['epsilon_xuv']},
    'H2O': {'nu_0': 4.835981008048 * 10**15 , 'mu_wind': 3, 'mu_plus_wind': 6, 'mu_photo': 18, "epsilon_xuv": wind_microphysics['H2O']['epsilon_xuv']},
    'CO2' : {'nu_0' : 3.3368 * 10**15, 'mu_wind': 7.2, 'mu_plus_wind': 14.4, 'mu_photo': (12+(16*2)), "epsilon_xuv": wind_microphysics['CO2']['epsilon_xuv']}, 
    'N2' : {'nu_0' : 3.7721 * 10**15, 'mu_wind': 7, 'mu_plus_wind': 14, 'mu_photo': 14*2, "epsilon_xuv": wind_microphysics['N2']['epsilon_xuv']},
    }

stellar_cases = [
    {'M_star': 1.40,  'age': 70.0 * 10**6, 'label': 'Young F-type Star (1.40 M$_\odot$, 70 Myr)'},
    {'M_star': 1.40,  'age': 1.0 * 10**9, 'label': 'Matured F-type Star (1.40 M$_\odot$, 1 Gyr)'},
    {'M_star': 1.0,  'age': 70.0 * 10**6, 'label': 'Young G-type Star (1.0 M$_\odot$, 70 Myr)'},
    {'M_star': 1.0,  'age': 1.0 * 10**9, 'label': 'Matured G-type Star (1.0 M$_\odot$, 1 Gyr)'},
    {'M_star': 0.7,  'age': 70.0 * 10**6, 'label': 'Young K-type Star (0.7 M$_\odot$, 70 Myr)'},
    {'M_star': 0.7,  'age': 1.0 * 10**9, 'label': 'Matured K-type Star (0.7 M$_\odot$, 1 Gyr)'}
]



In [ ]:
### Convergence test ###

resolutions = np.linspace(100, 10000, 100).astype(int)
convergence_data = {}
for species in species_microphysics.keys(): #for each species
    atms_varied_resolution = np.array([make_simple_atmosphere( 
        M_p =  8.41 * 5.9722 * 10**24,               #[kg] 
        F_xuv        = 10**2.93 * erg_to_joule * cm_to_m**(-2)  ,              #[kg s^-3]    
        R_p          = 2.73 * 6.371 * 10**6  ,          #[m]         planetary radius. 2 times earth radius
        P_0          = 2000   ,                      #[Pa]        pressure at the optical photosphere
        T_eq         = 567 ,                        #[K]         equilibrium temperature of the planet
        dominant_species=species,
        mu_photo=species_microphysics[species]['mu_photo'],
        resolution=res) for res in resolutions])
    
    # Process through your diagnostics framework
    M_dot_stitched, rr_mask, M_dot_rr, M_dot_el, _ = sweep_parameter(atms_varied_resolution)
    
    # Store arrays as numpy arrays inside the tracking dictionary
    convergence_data[species] = {
        'stitched': np.array(M_dot_stitched),
        'rr': np.array(M_dot_rr),
        'el': np.array(M_dot_el)
    }

In [ ]:
# 2. Render Convergence Graphic with Quantitative 1.5% Assessment
keys = list(convergence_data.keys())
colours = ['#228833', '#4477AA', '#EE6677', '#CCBB44']

# Initialize figure array grid
fig, axes = plt.subplots(len(keys), 1, figsize=(11, 3.5 * len(keys)), sharex=True)

# Main Title configured without bold text weights
fig.suptitle('Grid Convergence Analysis and 1.5% Stability Threshold Verification', fontsize=18, y=0.95)

if len(keys) == 1:
    axes = [axes]

print("\n--- QUANTITATIVE CONVERGENCE ASSESSMENT (1.5% THRESHOLD) ---")

for i, species in enumerate(keys):
    ax = axes[i]
    data = convergence_data[species]
    
    # --- FIXED: independent COMPONENT CONVERGENCE CHECK ---
    # Extract baseline final values from the highest resolution cap (last elements)
    final_el = data['el'][-1]
    final_rr = data['rr'][-1]
    
    # Absolute relative difference calculation arrays
    diffs_el = np.abs((data['el'] - final_el) / final_el)
    diffs_rr = np.abs((data['rr'] - final_rr) / final_rr)
    
    # --- FOR ENERGY-LIMITED ---
    stable_idx_el = len(resolutions) - 1
    while stable_idx_el >= 0 and diffs_el[stable_idx_el] <= 0.015:
        stable_idx_el -= 1
    stable_idx_el += 1
    res_needed_el = resolutions[stable_idx_el]
    
    # --- FOR RADIATION-RECOMBINATION ---
    stable_idx_rr = len(resolutions) - 1
    while stable_idx_rr >= 0 and diffs_rr[stable_idx_rr] <= 0.015:
        stable_idx_rr -= 1
    stable_idx_rr += 1
    res_needed_rr = resolutions[stable_idx_rr]
    
    # --- HIGHEST LIMIT EXTRACTION INTERSECTION ---
    if res_needed_el >= res_needed_rr:
        min_res_needed = res_needed_el
        limiting_regime = "Energy-Limited (EL)"
        final_diff_pct = diffs_el[stable_idx_el] * 100
    else:
        min_res_needed = res_needed_rr
        limiting_regime = "Radiation-Recombination (RR)"
        final_diff_pct = diffs_rr[stable_idx_rr] * 100
        
    print(f"Species: {species:<6} | Max N bounded by: {limiting_regime:<28} | Stabilizes at N = {min_res_needed} (Diff: {final_diff_pct:.2f}%)")
    
    # --- PLOTTING DATA LINES ---
    ax.plot(resolutions, data['el'], linestyle=':', color=colours[i], alpha=0.85, lw=2.0, 
            label=r'Energy-Limited ($\dot{M}_{\mathrm{EL}}$)')
    ax.plot(resolutions, data['rr'], linestyle='-', color=colours[i], alpha=0.85, lw=2.0, 
            label=r'Radiation-Recombination ($\dot{M}_{\mathrm{RR}}$)')
        
    # Shading reflects the maximum required resolution computed dynamically above
    ax.axvspan(min_res_needed, resolutions.max(), color='green', alpha=0.05, 
               label=f'Converged Area ($\geq$ {min_res_needed})')
    
    ax.axvline(min_res_needed, color='green', linestyle='--', alpha=0.3, lw=1.2)

    # Subplot Structural Formatting Setup
    ax.grid(True, alpha=0.25, linestyle=':')
    ax.set_yscale('log')
    
    all_vals = np.concatenate([data['el'], data['rr']])
    ax.set_ylim(all_vals.min() * 0.5, all_vals.max() * 2.0)
    
    ax.tick_params(axis='both', which='major', labelsize=13, length=6)
    ax.tick_params(axis='both', which='minor', labelsize=11, length=4)
    
    # Individual Subplot Legend Blocks
    ax.legend(loc='upper right', fontsize=14, frameon=True, facecolor='white', 
              edgecolor='#e0e0e0', title=f'Composition: {species}', title_fontsize=15)

# Configure the unified global labels on the main figure canvas
axes[-1].set_xlabel('Radial Grid Integration Steps [N]', fontsize=16, labelpad=10)
fig.supylabel(r'Mass-loss Rate [$\mathrm{kg\ s}^{-1}$]', fontsize=16, x=0.04)

print("----------------------------------------------------------")
plt.subplots_adjust(hspace=0, bottom=0.05, top=0.93, left=0.12, right=0.95)
plt.savefig('Keepers/convergence_analysis.png', dpi=300)
plt.show()

In [ ]:
### Need to plot whether the escape rate is rr_limited or not, but will run it only over planetary masses. Assuming GJ1214b like planet ###

Mass_array = np.linspace(0.9, 16.6, 100) * M_earth

atmospheres_swept_varied_mass_MR = {}
for species in species_microphysics.keys(): #for each species
    atms_varied_mass_MR = np.array([make_simple_atmosphere( #make at atmosphere where only the mass is varied, but the radius is determined by an MR relation
        M_p=m,
        F_xuv=10**2.93 * erg_to_joule * cm_to_m**(-2),
        mu_photo=species_microphysics[species]['mu_photo'],
        nu_0=species_microphysics[species]['nu_0'],
        mu_wind=species_microphysics[species]['mu_wind'],
        mu_plus_wind=species_microphysics[species]['mu_plus_wind'],
        P_0=2000,
        T_wind=10**4,
        T_eq=567,
        dominant_species=species,
        resolution=6000,
        determine_radius=True) for m in Mass_array])

    #now we have an array of atmospheres with different masses, we can sweep through them and get the mass loss rates and check if it is rr_limited
    M_dot_stitched, rr_mask, M_dot_rr, M_dot_el, transonic_mask = sweep_parameter(atms_varied_mass_MR)
    atmospheres_swept_varied_mass_MR[species] = (M_dot_stitched, rr_mask, M_dot_rr, M_dot_el, transonic_mask)



In [ ]:

keys = list(atmospheres_swept_varied_mass_MR.keys())

# Consolidated academic palette assignments:
# Fixed functional colors across all panels
color_el = '#4477AA'         # Blue for Energy-Limited
color_rr = '#EE6677'         # Red for Rad-Recomb
color_blowoff = 'cyan'    # cyan for Blow-off like later
color_transition = '#CCBB44' # Yellow-Gold for transition marker fills

# Set up subplots with hspace=0 inside gridspec_kw to make them touch like slabs
fig, axes = plt.subplots(len(keys), 1, figsize=(12, 4.5 * len(keys)), sharex=True, gridspec_kw={'hspace': 0})
fig.suptitle('Hydrodynamic Escape Rates and Regimes for Different Planetary \n Masses and Compositions for Planets with GJ1214b \n Incident XUV Flux and Equilibrium Temperature', fontsize=20, y=0.96)

masses = Mass_array / M_earth

# Ensure axes is always iterable even if only 1 species is passed
if len(keys) == 1:
    axes = [axes]

for i, species in enumerate(keys):
    ax = axes[i]
    
    # Unpack variables precisely matching: (R_base_array, R_sonic_array, M_dot_rr, M_dot_el, rr_mask)
    R_base_array, R_sonic_array, M_dot_rr, M_dot_el, rr_mask = atmospheres_swept_varied_mass_MR[species]
    
    # 1. Plot pure Energy-Limited curve (Dotted ':') using the global EL color
    ax.plot(masses, M_dot_el, linestyle=':', color=color_el, alpha=0.9, lw=3.0, label='Pure Energy-Limited ($\dot{M}_{\mathrm{EL}}$)')
    
    # 2. Plot pure Radiation-Recombination curve (Solid) using the global RR color
    ax.plot(masses, M_dot_rr, linestyle='solid', color=color_rr, alpha=0.9, lw=3.0, label='Pure Rad-Recomb ($\dot{M}_{\mathrm{RR}}$)')
    
    # 3. Introduce the Blow-off check via a -. line
    is_blowoff = R_sonic_array <= R_base_array
    if np.any(is_blowoff):
        blowoff_masses = masses[is_blowoff]
        blowoff_rates = M_dot_rr[is_blowoff]
        ax.scatter(blowoff_masses, blowoff_rates, linestyle='dotted', color=color_blowoff, alpha=0.9, lw=2.5, label='Blow-off')

    # 4. Find where EL and RR cross over
    idx_cross = np.where(np.diff(np.sign(M_dot_el - M_dot_rr)))[0]
    
    for idx in idx_cross:
        m1, m2 = masses[idx], masses[idx+1]
        y_el1, y_el2 = M_dot_el[idx], M_dot_el[idx+1]
        y_rr1, y_rr2 = M_dot_rr[idx], M_dot_rr[idx+1]
        
        denom = (np.log10(y_el2) - np.log10(y_el1)) - (np.log10(y_rr2) - np.log10(y_rr1))
        if denom != 0:
            fraction = (np.log10(y_rr1) - np.log10(y_el1)) / denom
            cross_mass = m1 + fraction * (m2 - m1)
            cross_val = 10**(np.log10(y_el1) + fraction * (np.log10(y_el2) - np.log10(y_el1)))
        else:
            cross_mass = (m1 + m2) / 2.0
            cross_val = (y_el1 + y_el2) / 2.0
            
        ax.plot(cross_mass, cross_val, marker='o', markersize=12, 
                markerfacecolor=color_transition, markeredgecolor='black', markeredgewidth=1.5, 
                linestyle='none', zorder=5, label='EL and RR Regime Transition')

    # Add species identity as an internal text box instead of an external subplot title
    fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in species])
    ax.text(0.1, 0.12, f'${fmt_species}$-dominated', fontsize=16, 
            transform=ax.transAxes, verticalalignment='bottom', 
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=4))
    
    # Grid and Scales
    ax.grid(True, alpha=0.3, linestyle=':', color='#666666', which='both')
    ax.set_yscale('log')
    ax.set_ylim(10**(-7), 10**9)
    #ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=18, labelpad=10)
    
    # Prune edge y-labels to prevent overlapping where panels join
    def y_formatter(y, pos):
        val = int(np.round(np.log10(y)))
        if (val == 9 and i != 0) or (val == -7 and i != len(keys) - 1):
            return ''
        return f'$10^{{{val}}}$'
        
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(y_formatter))
    
    # Tick formatting
    ax.tick_params(axis='both', which='major', labelsize=15, length=7, width=1.2, direction='in', top=True, right=True)
    ax.tick_params(axis='both', which='minor', labelsize=12, length=4, width=1.0, direction='in', top=True, right=True)
    
    # Unique legend per subplot
    ax.legend(loc='lower right', fontsize=15, frameon=True, facecolor='white', edgecolor='#e0e0e0', framealpha=0.9)

# Global X-axis label assigned only to the bottom-most plot panel
axes[-1].set_xlabel(r'Planetary Mass [$M_\oplus$]', fontsize=18, labelpad=12)
fig.supylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=18)

# Standardize axis frame limits across all subplots
for ax in axes:
    ax.set_xlim(masses.min(), masses.max())

# Adjust figure layout to accommodate the snug multi-panel format cleanly
plt.subplots_adjust(bottom=0.10, top=0.89, left=0.12, right=0.95)
plt.savefig('Keepers/escape_rates_mass_sweep_GJ1214b.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. Initialize two separate dictionaries to store the sweeping results
results_H_coeff = {}
results_true_coeff = {}

for species in species_microphysics.keys(): 
    # --- RUN 1: FORCING HYDROGEN RECOMBINATION COEFFICIENTS ---
    atms_H_coeff = [make_simple_atmosphere( 
        M_p=m,
        F_xuv=10**2.93 * erg_to_joule * cm_to_m**(-2),
        mu_photo=species_microphysics[species]['mu_photo'],
        nu_0=wind_microphysics[species]['nu_0'],
        mu_wind=wind_microphysics[species]['mu_wind'],
        mu_plus_wind=wind_microphysics[species]['mu_plus_wind'],
        rr_coeff=wind_microphysics['H2']['rr_coeff'], 
        P_0=2000,
        T_wind=10**4,
        T_eq=553,
        dominant_species=species,
        resolution=10000,
        determine_radius=True) for m in Mass_array]

    _, _, M_dot_rr_H, _, _ = sweep_parameter(atms_H_coeff)
    results_H_coeff[species] = np.array(M_dot_rr_H)

    # --- RUN 2: USING TRUE SPECIES RECOMBINATION COEFFICIENTS ---
    atms_true_coeff = [make_simple_atmosphere( 
        M_p=m,
        F_xuv=10**2.93 * erg_to_joule * cm_to_m**(-2),
        mu_photo=species_microphysics[species]['mu_photo'],
        nu_0=wind_microphysics[species]['nu_0'],
        mu_wind=wind_microphysics[species]['mu_wind'],
        mu_plus_wind=wind_microphysics[species]['mu_plus_wind'],
        rr_coeff=wind_microphysics[species]['rr_coeff'], 
        P_0=2000,
        T_wind=10**4,
        T_eq=553,
        dominant_species=species,
        resolution=10000,
        determine_radius=True) for m in Mass_array]

    _, _, M_dot_rr_T, _, _ = sweep_parameter(atms_true_coeff)
    results_true_coeff[species] = np.array(M_dot_rr_T)


# --- 2. HIGH-READABILITY THESIS PLOTTING ROUTINE ---
keys = list(species_microphysics.keys())
colours = ['#228833', '#4477AA', '#EE6677', '#CCBB44', '#9933A5'] 

# Subplots sharing the Y-axis scale to guarantee immediate baseline visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
fig.suptitle(r' Recombination Coefficient Sweep for Radiation-Recombination Mass-Loss Rates', 
             fontsize=21.5, y=0.95)
masses = Mass_array / M_earth

axes = [ax1, ax2]
datasets = [results_H_coeff, results_true_coeff]
titles = [r"Forced Hydrogen Coefficients for All Species", 
          r"Ion Abundance Weighted Species Dependent Coefficients"]

# Plot both subplots dynamically
for ax, data_dict, title in zip(axes, datasets, titles):
    ax.set_title(title, fontsize=18, pad=14, loc='center')
    
    for i, species in enumerate(keys):
        pure_rr_mass_loss = data_dict[species]
        
        # Consistent line weights, distinct coloring scheme
        ax.plot(masses, pure_rr_mass_loss, linestyle='solid', color=colours[i], 
                label=rf'{species}', lw=3.5, alpha=0.95, zorder=3)

    # Format the individual axes elements
    ax.grid(True, alpha=0.6, linestyle=':', color='#777777', zorder=1)
    
   
    
    # Scale tick marks for publication size reduction safety
    ax.tick_params(axis='both', which='major', labelsize=14, length=6, width=1.2)
    ax.tick_params(axis='both', which='minor', labelsize=12, length=4, width=0.8)
    ax.set_xlim(masses[0], masses[-1])

fig.text(0.5, 0.11, r'Planetary Mass [$M_\oplus$]', fontsize=16, ha='center', va='center')

# Set the primary shared y-axis properties only on the left-most axis
ax1.set_ylabel(r'Mass-loss Rate [$\mathrm{kg\ s}^{-1}$]', fontsize=16, labelpad=10)
ax1.set_ylim(10**(-5), 10**9)
ax1.set_yscale('log')

# --- CREATING A UNIQUE SHARED GLOBAL LEGEND ---
# Extract lines and labels from the first axis to build a unified legend bar at the bottom
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles=handles, labels=labels, loc='lower center', bbox_to_anchor=(0.5, 0.02),
           ncol=len(keys), fontsize=16, frameon=True, facecolor='white', edgecolor='#d0d0d0')

# Padding layout adjustment to securely reserve empty space at the bottom for the shared legend
plt.tight_layout(rect=[0, 0.14, 1, 0.92])

# Define an ellipse using Figure coordinates (independent of the log-scale warp)
# xy=(center_x, center_y), width, height
highlight_circle = patches.Ellipse(
    xy=(0.79, 0.47),        # 0.5 is horizontally centered, 0.6 is slightly above middle
    width=0.15,           # Width as a fraction of the total figure width
    height=0.25,          # Height as a fraction of the total figure height
    angle=180,             # Tilt angle in degrees if you want it diagonal
    edgecolor='firebrick',# High-contrast color for thesis readability
    facecolor='none',     # Transparent inside
    linewidth=2.5,        # Sharp line weight
    linestyle='-',        # Solid line
    transform=fig.transFigure, # Dictates that we are using global Figure coordinates
    zorder=5              # Ensures it sits strictly on top of lines and grids
)

# Add the patch directly to the figure canvas
fig.patches.append(highlight_circle)
fig.text(0.79, 0.32, r'RR Rate Change', fontsize=14, ha='center', va='center', color='firebrick')

pointer_line = patches.ConnectionPatch(
    xyA=(0.35, 0.45),       # Start point (e.g., where you might put a text label)
    xyB=(0.7, 0.45),      # End point (touching the top edge of your oblong circle)
    coordsA=fig.transFigure,
    coordsB=fig.transFigure,
    color='firebrick',
    linewidth=2,
    linestyle='--',
    arrowstyle="-|>"      # Adds a neat, professional arrowhead pointing at the region
)

fig.patches.append(pointer_line)
# plt.savefig('Keepers/Mass_sweep_MR_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# --- 1. COMPUTE THE RELATIVE DIFFERENCE RATIO ---
keys = list(species_microphysics.keys())
colours = ['#228833', '#4477AA', '#EE6677', '#CCBB44'] # Standardized color scheme
masses = Mass_array / M_earth

# FIXED: Custom properties configured so that H2 and N2 do not completely overwrite each other
linestyles = {
    'H2': 'solid',
    'H2O': 'solid',
    'CO2': 'solid',
    'N2': 'solid'         # Dash-dot pattern cuts across solid tracks cleanly
}

line_widths = {
    'H2': 6.0,         # Double the width of H2 so it forms a visual background track
    'H2O': 3.5,
    'CO2': 3.5,
    'N2': 3.0          # Slightly thinner line allows the underlying H2 track to act as an outline
}

z_orders = {
    'H2': 3,          # Base profile layer
    'H2O': 4,
    'CO2': 4,
    'N2': 5           # Draw N2 on top so its dash-dots break up the solid green underneath
}

# Initialize a standard single-panel figure optimized for thesis dimensions
fig, ax = plt.subplots(figsize=(10, 7))

fig.suptitle(f'Impact of Ion Abundance Weighted (IAW) Radiation Recombination \n Coefficients  on RR Mass Loss Rates', 
             fontsize=19, y=0.96)

# --- 2. PLOT THE DEVIATION RATIOS WITH VISIBILITY FIXES ---
for i, species in enumerate(keys):
    ratio = results_true_coeff[species] / results_H_coeff[species]
    
    # Format the species label string for clean standard LaTeX subscripts
    formatted_label = species
    for num in ['2', '3', '4']:
        formatted_label = formatted_label.replace(num, f'_{num}')
        
    ax.plot(masses, ratio, 
            linestyle=linestyles.get(species, 'solid'), 
            color=colours[i], 
            label=rf'$\mathrm{{{formatted_label}}}$', 
            lw=line_widths.get(species, 3.5), 
            alpha=0.95, 
            zorder=z_orders.get(species, 3))

# --- 3. CRITICAL BASELINE REFERENCE LINE ---
ax.axhline(1.0, linestyle=':', color='#333333', linewidth=1.2, alpha=0.7, zorder=2)
#ax.text(masses[-1] * 0.98, 0.5, r'Hydrogen is the baseline $\dot{M}_{\mathrm{H}}$', fontsize=11, color='#555555', ha='right', va='bottom', style='italic')

# --- 4. AXES FORMATTING & SCALING (With Pure Linear Tick Labels) ---
ax.set_xlabel(r'Planetary Mass [$M_\oplus$]', fontsize=16, labelpad=10)
ax.set_ylabel(r'Mass-loss Rate Ratio [$\dot{M}_{\alpha_B = \mathrm{IAW}}\ /\ \dot{M}_{\alpha_B = \mathrm{H}}$]', fontsize=16, labelpad=10)

# Maintained exact requested layout bounds on linear scale
ax.set_ylim(0, 5) 
ax.set_xlim(masses[0], masses[-1])

# FIXED: Force a major tick locator at every integer interval to show 1, 3, 5, 7, 9 explicitly
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

# Standardized linear axis text formatter that avoids log operations 
def linear_tick_formatter(y, pos):
    if y == 0:
        return "0"
    elif y.is_integer():
        return f"{int(y)}"  # Formats standard integers cleanly, keeping 10 exactly as "10"
    else:
        return f"{y:.1f}"   # Preserves decimals uniformly if non-integers are captured

ax.yaxis.set_major_formatter(ticker.FuncFormatter(linear_tick_formatter))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())

# Format clean grid lines and ticks
ax.grid(True, which='both', alpha=0.4, linestyle=':', color='#777777', zorder=1)
ax.tick_params(axis='both', which='major', labelsize=12, length=6, width=1.2)
ax.tick_params(axis='both', which='minor', labelsize=10, length=4, width=0.8)

# --- 5. LEGEND PLACEMENT ---
ax.legend(loc='upper right', fontsize=15, frameon=True, facecolor='white', edgecolor='#d0d0d0', shadow=False)
plt.savefig('Keepers/RR_coeff_comparison.png', dpi=300)
plt.show()

In [ ]:
max_pct_diffs = {}

for key in keys:
    # Use the true species coefficient run as the baseline "ground truth"
    baseline = results_true_coeff[key]
    
    # Avoid potential division by zero if any rates are exactly 0
    with np.errstate(divide='ignore', invalid='ignore'):
        pct_diff = np.abs(results_H_coeff[key] - baseline) / baseline * 100
        # Replace NaNs or Infs with 0 if they occur at the lower boundary
        pct_diff = np.nan_to_num(pct_diff, nan=0.0, posinf=0.0)
    
    max_pct_diffs[key] = np.max(pct_diff)
    print(f"Species: {key:<6} | Maximum deviation from true value: {max_pct_diffs[key]:.2f}%")

# Find the absolute peak percentage change across the entire study
overall_max_pct = np.max(list(max_pct_diffs.values()))
print(f"\nOverall maximum impact of switching coefficients: {overall_max_pct:.2f}%")

In [ ]:
print("--- AVERAGE IMPACT ANALYSIS (ORDERS OF MAGNITUDE SHIFT) ---")
mean_log_diffs = {}

for key in keys:
    # Avoid log10(0) issues by filtering out or handling extremely small unphysical numbers
    log_H = np.log10(np.clip(results_H_coeff[key], 10**-20, None))
    log_T = np.log10(np.clip(results_true_coeff[key], 10**-20, None))
    
    # Absolute difference in log-space represents a factor change
    log_diff = np.abs(log_H - log_T)
    
    mean_log_diffs[key] = np.mean(log_diff)
    # 10**mean_log_diffs gives the average multiplicative factor change
    factor_change = 10**mean_log_diffs[key]
    
    print(f"Species: {key:<6} | Average log10 shift: {mean_log_diffs[key]:.3f} dex (Factor of {factor_change:.2f}x)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10,6))
fig.suptitle('Mass-radius relation (adapted from Parc et. al. 2024)')
#Setting up the masses and radii we want to plot the relation of
R_earth = 6371000 #[m] mean earth radius
MR_masses = np.logspace(-0.3, 4, 1000) #[M_earth]
radii = np.array([Atmosphere.determine_radius_from_MR_relation(m)/R_earth for m in (MR_masses * M_earth)])

#Creating the masks
low_mass_mask = MR_masses < 10 #needs to be divided by earth masses again because the comparison is in earth masses
intermediate_mass_mask =  (MR_masses >= 10) & (MR_masses < 138)
high_mass_mask = MR_masses >= 138

#plotting the masks
ax.scatter(MR_masses[low_mass_mask], radii[low_mass_mask], label='R = $1.02 \cdot M_{{p,\oplus}}^{0.28}$', color='#0072b2')
ax.scatter(MR_masses[intermediate_mass_mask], radii[intermediate_mass_mask], label='R = $0.61 \cdot M_{{p,\oplus}}^{0.67}$', color='#cc79a7')
ax.scatter(MR_masses[high_mass_mask], radii[high_mass_mask], label='R = $11.9 \cdot M_{{p,\oplus}}^{0.01}$', color='#009e73')

#Switching axes to log scales
ax.set_xscale('log')
ax.set_yscale('log')

#Defineing and forcing the exact tick marks seen in the paper by Parc et. al. 2024
x_ticks = [1, 5, 10, 20, 50, 100, 1000, 10000]
ax.set_xticks(x_ticks, labels=[str(x) for x in x_ticks])

y_ticks = [1, 2, 3, 5, 10]
ax.set_yticks(y_ticks, labels=[str(y) for y in y_ticks])

#Sets limits to frame the data like the paper's plot bounds
ax.set_xlim(0.2, 12000)
ax.set_ylim(0.5, 20)
ax.set_xlabel('Mass [$M_\oplus$]')
ax.set_ylabel('Radius [$R_\oplus$]')
ax.grid()
ax.legend()
#plt.savefig('Keepers/MR_relation.png', dpi=300)
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(11, 8.5)) # Slightly expanded width to balance large boxes

# Setting up the masses and radii we want to plot the relation of
R_earth = 6371000 #[m] mean earth radius
MR_masses = np.logspace(-0.3, 4, 1000) #[M_earth]
radii = np.array([Atmosphere.determine_radius_from_MR_relation(m)/R_earth for m in (MR_masses * M_earth)])

# --- NUMERICAL EXTRACTION OF BOUNDARY MASSES ---
mass_at_1_0 = np.interp(1.0, radii, MR_masses)
mass_at_1_8 = np.interp(1.8, radii, MR_masses)
mass_at_4_0 = np.interp(4.0, radii, MR_masses)

# Defining plot absolute limits early to construct the background bounding geometry
x_min, x_max = 0.2, 12000
y_min, y_max = 0.5, 20

# --- SHADING THE L-SHAPED SUPER-EARTH REGION ---
# FIXED: Changed 'darkgeometry' to 'darkgreen'
se_horiz_box = patches.Polygon([
    [x_min, 1.0], [mass_at_1_8, 1.0], 
    [mass_at_1_8, 1.7], [x_min, 1.7]
], closed=True, facecolor='green', alpha=0.10, edgecolor='darkgreen', linewidth=1.5, zorder=1)

se_vert_box = patches.Polygon([
    [mass_at_1_0, y_min], [mass_at_1_8, y_min], 
    [mass_at_1_8, 1.0], [mass_at_1_0, 1.0]
], closed=True, facecolor='green', alpha=0.10, edgecolor='none', zorder=1)

ax.add_patch(se_horiz_box)
ax.add_patch(se_vert_box)

# --- SHADING THE L-SHAPED SUB-NEPTUNE REGION ---
sn_horiz_box = patches.Polygon([
    [x_min, 1.7], [mass_at_4_0, 1.7], 
    [mass_at_4_0, 4.0], [x_min, 4.0]
], closed=True, facecolor='blue', alpha=0.12, edgecolor='darkblue', linewidth=1.5, zorder=1)

sn_vert_box = patches.Polygon([
    [mass_at_1_8, y_min], [mass_at_4_0, y_min], 
    [mass_at_4_0, 1.7], [mass_at_1_8, 1.7]
], closed=True, facecolor='blue', alpha=0.12, edgecolor='none', zorder=1)

ax.add_patch(sn_horiz_box)
ax.add_patch(sn_vert_box)

# --- PLOTTING RELATION LINES (Thicker Line Weights) ---
low_mass_mask = MR_masses < 10 
intermediate_mass_mask = (MR_masses >= 10) & (MR_masses < 138)
high_mass_mask = MR_masses >= 138

line1, = ax.plot(MR_masses[low_mass_mask], radii[low_mass_mask], 
                 label=r'R = $1.02 \cdot M_{\mathrm{p},\oplus}^{0.28}$', color='#0072b2', lw=5.5, zorder=3)
line2, = ax.plot(MR_masses[intermediate_mass_mask], radii[intermediate_mass_mask], 
                 label=r'R = $0.61 \cdot M_{\mathrm{p},\oplus}^{0.67}$', color='#cc79a7', lw=5.5, zorder=3)
line3, = ax.plot(MR_masses[high_mass_mask], radii[high_mass_mask], 
                 label=r'R = $11.9 \cdot M_{\mathrm{p},\oplus}^{0.01}$', color='#009e73', lw=5.5, zorder=3)

# --- VISUAL BOUNDARY CORNERS ---
ax.plot([x_min, mass_at_1_0, mass_at_1_0], [1.0, 1.0, y_min], 
        color='navy', linestyle=':', alpha=0.4, lw=1.5, zorder=2)
ax.plot([x_min, mass_at_1_8, mass_at_1_8], [1.7, 1.7, y_min], 
        color='dimgray', linestyle=':', alpha=0.6, lw=1.8, zorder=2)
ax.plot([x_min, mass_at_4_0, mass_at_4_0], [4.0, 4.0, y_min], 
        color='dimgray', linestyle='--', alpha=0.6, lw=1.8, zorder=2)

# --- AXIS SCALES & BOUNDS ---
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# --- GRAPHICS CLEANUP & TYPOGRAPHY ---
ax.set_title('Planetary Mass-Radius Relation for super-Earths \n and sub-Neptunes (Adapted from Parc et al. 2024)', fontsize=20, pad=20)
ax.set_xlabel(r'Mass [$M_\oplus$]', fontsize=18, labelpad=12)
ax.set_ylabel(r'Radius [$R_\oplus$]', fontsize=18, labelpad=12)

x_ticks = [1, 5, 10, 20, 50, 100, 1000, 10000]
ax.set_xticks(x_ticks, labels=[str(x) for x in x_ticks])
y_ticks = [0.5, 1, 2, 3, 5, 10, 20]
ax.set_yticks(y_ticks, labels=[str(y) for y in y_ticks])

ax.tick_params(axis='both', which='major', labelsize=15, length=8, width=1.5)
ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)

ax.grid(True, which='both', linestyle=':', alpha=0.3, color='gray', zorder=0)
ax.set_axisbelow(True)

# --- CUSTOM MULTI-LINE LEGEND TEXT GENERATION ---
se_legend_label = (f'Super-Earth Regime Limits:\n'
                   f'• Radii: $1.0 - 1.8\ R_\oplus$\n'
                   f'• Masses: {mass_at_1_0:.1f} - {mass_at_1_8:.1f} $M_\oplus$')
se_horiz_box.set_label(se_legend_label)

sn_legend_label = (f'Sub-Neptune Regime Limits:\n'
                   f'• Radii: $1.8 - 4.0\ R_\oplus$\n'
                   f'• Masses: {mass_at_1_8:.1f} - {mass_at_4_0:.1f} $M_\oplus$')
sn_horiz_box.set_label(sn_legend_label)


# =============================================================================
# OPTIMIZED: ULTRA-LARGE LEGEND BOX PLACEMENTS WITH COMPACT INTERIOR SPACING
# =============================================================================

# Legend 1: Equations (Placed in Upper Left with bumped spacing properties)
legend1 = ax.legend(handles=[line1, line2, line3], title="Empirical M-R Scaling", 
                    fontsize=16, title_fontsize=20, loc='upper left', 
                    borderpad=0.9, labelspacing=0.8, handletextpad=1.0,
                    frameon=True, facecolor='#fbfbfb', edgecolor='#b0b0b0')

# Add Legend 1 manually as a persistent artist so the next call won't overwrite it
ax.add_artist(legend1)

# Legend 2: Demographic Zones (Placed in Lower Right with matching spacing formats)
legend2 = ax.legend(handles=[se_horiz_box, sn_horiz_box], title="Planetary Classification",
                    fontsize=15, title_fontsize=20, loc='lower right', 
                    borderpad=1.0, labelspacing=1.1, handletextpad=1.0,
                    frameon=True, facecolor='#fbfbfb', edgecolor='#b0b0b0')


# =============================================================================

plt.tight_layout()
plt.savefig('Keepers/MR_relation_improved.png', dpi=300)
plt.show()
plt.close(fig)

In [ ]:
### Want to see how the gravitational well changes with the MR relation ###

### compare with salz et al 2016 if can host hydrodynamic escape ###
G = sp.constants.G #[m^3 kg^-1 s^-2]

grav_pot_SI = np.array([- G * m / Atmosphere.determine_radius_from_MR_relation(m) for m in (MR_masses * M_earth)]) #[m^2 s^-2] gravitational potential at the planetary radius, where G: gravitational constant, M_p: planetary mass, R_p: planetary radius
grav_pot_cgs = grav_pot_SI * cm_to_m**(-2) #[cm^2 s^-2] gravitational potential at the planetary radius in cgs units, where grav_pot_SI: gravitational potential at the planetary radius in SI units
grav_compare = np.log10(-grav_pot_cgs) #[log10(cm^2 s^-2)] logarithm of the negative gravitational potential at the planetary radius in cgs units, where grav_pot_cgs: gravitational potential at the planetary radius in cgs units
salz_strong_grav_threshold = 13.6 #Larger than this value and gravity is too high for hydrodynamic escape. No EL and no wind for RR. REFERENCE Salz et. al. 2016
salz_weak_grav_threshold = 13.11 #Smaller than this value and the planet can host hydrodynamic wind and therefore EL and RR.

fig, ax = plt.subplots(1, 1, figsize=(6, 8))
fig.suptitle('Gravitational well of MR relation W.R.T. Salz et. al. 2016 regimes')

#Masks for the three possible regimes for the atmospheres:
weak_grav_mask = grav_compare < salz_weak_grav_threshold
strong_grav_mask = grav_compare > salz_strong_grav_threshold
intermediate_mask = (grav_compare >= salz_weak_grav_threshold) & (grav_compare <= salz_strong_grav_threshold)

ax.scatter(MR_masses[weak_grav_mask], grav_compare[weak_grav_mask], 
           color='#228833', s=40, zorder=3, label='Weak Gravity (Wind Allowed)')

ax.scatter(MR_masses[intermediate_mask], grav_compare[intermediate_mask], 
           color='#CCBB44', s=40, zorder=3, label='Intermediate Regime (Declining Wind)')

ax.scatter(MR_masses[strong_grav_mask], grav_compare[strong_grav_mask], 
           color='#EE6677', s=40, zorder=3, label='Strong Gravity (No Hydrodynamic Escape)')

#Visual Enhancements: Add Threshold Lines and Shading 
ax.axhline(salz_weak_grav_threshold, color='black', linestyle=':', alpha=0.6, zorder=2)
ax.axhline(salz_strong_grav_threshold, color='black', linestyle=':', alpha=0.6, zorder=2)

# Soft background color shading for each area
ax.axhspan(ax.get_ylim()[0], salz_weak_grav_threshold, color='#228833', alpha=0.1, zorder=1)
ax.axhspan(salz_weak_grav_threshold, salz_strong_grav_threshold, color='#CCBB44', alpha=0.1, zorder=1)
ax.axhspan(salz_strong_grav_threshold, ax.get_ylim()[1], color='#EE6677', alpha=0.1, zorder=1)

#5 Labels and Formatting
ax.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_xlabel(r'Planetary Mass [$M_\oplus$]')
ax.set_ylabel(r'$\log_{10}(-\Phi_\mathrm{G})$ [$\log_{10}(\mathrm{cm}^2\,\mathrm{s}^{-2})$]')
ax.semilogx()
x_ticks = [1, 5, 10, 20, 50, 100, 1000, 10000]
ax.set_xticks(x_ticks, labels=[str(x) for x in x_ticks])
# Place the legend in a clean spot
ax.legend(loc='best', frameon=True, shadow=False, facecolor='white', edgecolor='none')
ax.grid(True)
#plt.savefig('Keepers/GravPotential_vs_Regime_MR_relation.png', dpi=300)
plt.show()
plt.close()

In [ ]:
G = sp.constants.G #[m^3 kg^-1 s^-2]

grav_pot_SI = np.array([- G * m / Atmosphere.determine_radius_from_MR_relation(m) for m in (MR_masses * M_earth)]) #[m^2 s^-2]
grav_pot_cgs = grav_pot_SI * cm_to_m**(-2) #[cm^2 s^-2]
grav_compare = np.log10(-grav_pot_cgs) #[log10(cm^2 s^-2)]

salz_strong_grav_threshold = 13.6 
salz_weak_grav_threshold = 13.11 

# --- NUMERICAL EXTRACTION OF RELEVANT BOUNDARY VALUES ---
# Find the exact gravitational potential at your mass limits
grav_at_0_9 = np.interp(0.9, MR_masses, grav_compare)
grav_at_16_6 = np.interp(16.6, MR_masses, grav_compare)

# Establish dynamic axis plot limits early to securely align background geometries
x_min, x_max = 0.2, 12000
y_min, y_max = 11.5, 14.8

# Initialize figure with publication aspect ratio
fig, ax = plt.subplots(1, 1, figsize=(8, 9))
fig.suptitle('MR Relation vs. Salz et al. 2016 Wind Regimes', fontsize=17, y=0.96)

# --- SHADING THE L-SHAPED STUDY SCOPE REGION (0.9 to 16.6 M_earth) ---
# 1. Horizontal block extending from the left edge (x_min) to the upper mass boundary (16.6)
#    spanning vertically from the lowest potential (grav_at_0_9) up to the highest potential (grav_at_16_6)
scope_horiz_box = patches.Polygon([
    [x_min, grav_at_0_9], [16.6, grav_at_0_9], 
    [16.6, grav_at_16_6], [x_min, grav_at_16_6]
], closed=True, color='blue', alpha=0.08, zorder=1)

# 2. Vertical block extending from the bottom axis floor (y_min) up to the lower potential edge
#    spanning horizontally across the exact mass window (0.9 to 16.6 M_earth)
scope_vert_box = patches.Polygon([
    [0.9, y_min], [16.6, y_min], 
    [16.6, grav_at_0_9], [0.9, grav_at_0_9]
], closed=True, color='blue', alpha=0.08, zorder=1)

ax.add_patch(scope_horiz_box)
ax.add_patch(scope_vert_box)

# --- MASKS FOR REGIME CLASSIFICATIONS ---
weak_grav_mask = grav_compare < salz_weak_grav_threshold
strong_grav_mask = grav_compare > salz_strong_grav_threshold
intermediate_mask = (grav_compare >= salz_weak_grav_threshold) & (grav_compare <= salz_strong_grav_threshold)

# Scatter distributions
ax.scatter(MR_masses[weak_grav_mask], grav_compare[weak_grav_mask], 
           color='#228833', s=45, zorder=3, label='Weak Gravity (Wind Allowed)')

ax.scatter(MR_masses[intermediate_mask], grav_compare[intermediate_mask], 
           color='#CCBB44', s=45, zorder=3, label='Intermediate Regime (Declining Wind)')

ax.scatter(MR_masses[strong_grav_mask], grav_compare[strong_grav_mask], 
           color='#EE6677', s=45, zorder=3, label='Strong Gravity (No Hydrodynamic Escape)')

# --- VISUAL ENHANCEMENTS: LINES AND BASE SHADING ---
ax.axhline(salz_weak_grav_threshold, color='black', linestyle=':', alpha=0.6, zorder=2)
ax.axhline(salz_strong_grav_threshold, color='black', linestyle=':', alpha=0.6, zorder=2)

# Soft background color shading for each area behind the main focus box
ax.axhspan(y_min, salz_weak_grav_threshold, color='#228833', alpha=0.06, zorder=0)
ax.axhspan(salz_weak_grav_threshold, salz_strong_grav_threshold, color='#CCBB44', alpha=0.06, zorder=0)
ax.axhspan(salz_strong_grav_threshold, y_max, color='#EE6677', alpha=0.06, zorder=0)

# Inner dashed guide lines detailing the L-shape borders
ax.plot([x_min, 0.9, 0.9], [grav_at_0_9, grav_at_0_9, y_min], 
        color='navy', linestyle=':', alpha=0.5, lw=1.2, zorder=2)
ax.plot([x_min, 16.6, 16.6], [grav_at_16_6, grav_at_16_6, y_min], 
        color='navy', linestyle='--', alpha=0.5, lw=1.2, zorder=2)

# --- LABELS AND FORMATTING ---
ax.set_xlabel(r'Planetary Mass [$M_\oplus$]', fontsize=14, labelpad=8)
ax.set_ylabel(r'$\log_{10}(-\Phi_\mathrm{G})$ [$\log_{10}(\mathrm{cm}^2\,\mathrm{s}^{-2})$]', fontsize=14, labelpad=8)
ax.set_xscale('log')

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

x_ticks = [1, 5, 10, 20, 50, 100, 1000, 10000]
ax.set_xticks(x_ticks, labels=[str(x) for x in x_ticks])
ax.tick_params(axis='both', which='major', labelsize=11, length=6, width=1.2)
ax.tick_params(axis='both', which='minor', length=4, width=0.8)

ax.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)

# Generate specific legend tracking metadata for the consideration scope
scope_legend_label = (f'This Study Consideration Scope:\n'
                      f'Masses: $0.9 - 16.6\ M_\oplus$\n')
scope_horiz_box.set_label(scope_legend_label)

# Place the legend in a clean spot with a crisp boundary frame
ax.legend(loc='upper left', frameon=True, facecolor='#fbfbfb', edgecolor='#d3d3d3', fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('Keepers/GravPotential_vs_Regime_MR_relation.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# --- 1. PARAMETER SWEEP OVER WIND BASE PRESSURES ---

# Using geomspace ensures your 3000 steps are evenly distributed across log decades 
# from 10^-2 Pa down to 10^-6 Pa
P_base_array = np.geomspace(10**(-2), 10**(-6), 3000) 

atmospheres_swept_P_base = {}
for species in species_microphysics.keys(): 
    atm_varied_P_base = [make_simple_atmosphere(
        M_p=8.41 * 5.9722 * 10**24,
        R_p=2.73 * 6.371 * 10**6,
        F_xuv=10**2.93 * erg_to_joule * cm_to_m**(-2),
        nu_0=species_microphysics[species]['nu_0'],
        mu_wind=species_microphysics[species]['mu_wind'],
        mu_plus_wind=species_microphysics[species]['mu_plus_wind'],
        T_eq=567,
        mu_photo=species_microphysics[species]['mu_photo'],
        P_base=P_base,
        T_wind=10**4,
        dominant_species=species,
    ) for P_base in P_base_array]
    
    # FIXED: Extracting ONLY the pure theoretical Rad-Recomb rates (M_dot_rr)
    _, _, M_dot_rr_P, _, _ = sweep_parameter(atm_varied_P_base)
    atmospheres_swept_P_base[species] = np.array(M_dot_rr_P)

keys = list(atmospheres_swept_P_base.keys())


# --- 2. HIGH-READABILITY THESIS PLOTTING ROUTINE ---

colours = ['#228833', '#4477AA', '#EE6677', '#CCBB44', '#9933A5']
fig, ax = plt.subplots(1, 1, figsize=(11, 8))
fig.suptitle(r'Sensitivity of Radiation-Recombination Escape Rate to Base Pressure Choice ', 
             fontsize=18, y=0.96)

# Plot each species as a single, uninterrupted continuous line
for i, species in enumerate(keys):
    pure_rr_mass_loss = atmospheres_swept_P_base[species]
    ax.plot(P_base_array, pure_rr_mass_loss, linestyle='solid', color=colours[i], 
            label=rf'{species}', lw=3.5, alpha=0.95, zorder=3)

# Figure Formatting for Thesis/Publication
ax.grid(True, alpha=0.6, linestyle=':', color='#777777', zorder=1)

# Pure LaTeX, un-bolded formatting for axis strings
ax.set_xlabel(r'Wind Base Pressure [$\mathrm{Pa}$]', fontsize=15, labelpad=10)
ax.set_ylabel(r'Radiation-Recombination Escape Rate[$\mathrm{kg\ s}^{-1}$]', fontsize=15, labelpad=10)

# Set both scales to log to observe potential power-law relationships cleanly
ax.set_xscale('log')
ax.set_yscale('log')

# Explicitly defining limits based on your calculation constraints
ax.set_xlim(P_base_array[0], P_base_array[-1])
ax.set_ylim(10**(0), 10**9)

# Clean, distinct tick formatting for log grids
ax.tick_params(axis='both', which='major', labelsize=14, length=7, width=1.2)
ax.tick_params(axis='both', which='minor', labelsize=12, length=4, width=0.8)

# Legend clean up
ax.legend(loc='best', fontsize=13, frameon=True, facecolor='white', edgecolor='#d0d0d0')

plt.tight_layout(rect=[0, 0, 1, 0.93])
#plt.savefig('Keepers/P_base_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- GLOBAL GRID VARIABLES ---
semi_major_axis = np.logspace(-2, 0, 90) * au  # Distance from 0.01 to 10 AU
masses_2D = np.linspace(0.9, 16.6, 90) * M_earth   # Planet mass from 0.5 to 30 M_earth

n_masses = len(masses_2D)
n_distances = len(semi_major_axis)
masses_plot = masses_2D / M_earth

# Master master repository to hold data for every species
computed_data_all_species = {}

print("Beginning master multi-species, multi-case stellar evolution sweep...")

# 1. OUTER LOOP: Iterate over all species present in your microphysics dictionary
for species_name, specs in species_microphysics.items():
    print(f"\n========================================================")
    print(f" PROCESSING ATMOSPHERIC COMPOSITION: {species_name}")
    print(f"========================================================")
    
    # Initialize the data container list specifically for this composition
    computed_data_all_species[species_name] = []
    
    # 2. INNER LOOP: Run the multi-case stellar evolution sweep for this specific composition
    for case in stellar_cases:
        M_s = case['M_star'] * solar_mass
        age_s = case['age']
        print(f" -> Running Grid: M_star = {case['M_star']} M_sun | Age = {age_s/1e6:.1f} Myr")
        
        # Calculate stellar fluxes corresponding to the distance array
        F_bol_array, F_xuv_array = get_escape_fluxes(M_s, age_s, semi_major_axis)
        
        fluxes_plot = F_xuv_array / F_xuv_earth
        X, Y = np.meshgrid(masses_plot, fluxes_plot)
        
        # Run the computational grid (Resolution optimized to 6000)
        atms_varied_2D = np.array([[make_simple_atmosphere(
            M_p=M, 
            determine_radius=True, 
            F_xuv=F_xuv,
            nu_0=specs['nu_0'],
            mu_wind=specs['mu_wind'],
            mu_plus_wind=specs['mu_plus_wind'],
            mu_photo=specs['mu_photo'],         
            P_0=2000, 
            T_wind=10**4,
            T_eq=Atmosphere.determine_temperature_from_bolometric_flux(F_bol=F_b),
            resolution=6000, 
            dominant_species=species_name
        ) for F_xuv, F_b in zip(F_xuv_array, F_bol_array)] for M in masses_2D])

        # Flatten and sweep the parameter space
        atms_varied_flat = atms_varied_2D.flatten()
        
        # Unpack theoretical criteria values according to the structural formalism
        M_dot_flat, rr_mask_flat, _, _, transonic_mask_flat = sweep_parameter(atms_varied_flat)

        # Reconstruct 2D matrices (Reshape then transpose to match X, Y meshgrids)
        Z_mdot = np.array(M_dot_flat).reshape((n_masses, n_distances)).T
        Z_mask = np.array(rr_mask_flat).reshape((n_masses, n_distances)).T.astype(float)
        Z_transonic = np.array(transonic_mask_flat).reshape((n_masses, n_distances)).T.astype(float)
        
        # Store results into the specific species key list inside the master repository
        computed_data_all_species[species_name].append({
            'case': case, 
            'X': X, 
            'Y': Y, 
            'Z_mdot': Z_mdot, 
            'Z_mask': Z_mask, 
            'Z_transonic': Z_transonic,
            'fluxes_plot': fluxes_plot
        })

print("\n========================================================")
print("MASTER MULTI-SPECIES PARAMETER SWEEP COMPLETE.")
print("========================================================")

In [ ]:
# ====================================================================
# 1. SETUP TARGET CONFIGURATION (Optimized Layout & Sizing)
# ====================================================================
target_species = 'H2'  # e.g., 'H2O', 'CO2', 'H2', 'N2'
data_to_plot = computed_data_all_species[target_species]

num_plots = len(data_to_plot)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

# Figure setup: Slightly wider to prevent label clipping on the right
fig, axes = plt.subplots(num_rows, num_cols, figsize=(17.5, 5.8 * num_rows), sharex=True, sharey=False)
axes = np.atleast_1d(axes).flatten()

# ====================================================================
# 2. ITERATE AND RENDER PLOTS
# ====================================================================
for i, data in enumerate(data_to_plot):
    ax = axes[i]
    
    # Direct dictionary variable unpacking
    case, X, Y, Z_mdot, fluxes_plot = data['case'], data['X'], data['Y'], data['Z_mdot'], data['fluxes_plot']
    Z_el_mask = (~data['Z_mask'].astype(bool)).astype(float)
    Z_transonic = data['Z_transonic']
    
    # Base configuration: Flux coordinate system
    ax.set_yscale('log')
    ax.set_ylim(fluxes_plot.min(), fluxes_plot.max())
    
    # Primary continuous mass loss field mapping (FIXED FOR PDF SEAMLESS RENDERING)
    pcm = ax.pcolormesh(X, Y, Z_mdot, 
                        norm=colors.LogNorm(vmin=1e-2, vmax=1e11), 
                        cmap='plasma', 
                        shading='auto', 
                        edgecolors='face', # Closes the vector seams
                        linewidth=0)       # Ensures no artificial borders are drawn

    # FIXED: True transparent hatching layer using collection overrides instead of contourf
    el_contour = ax.contour(X, Y, Z_el_mask, levels=[0.5], colors='white', linewidths=2.5)
    
    # Generate an isolated hatching polygon path that is completely transparent on its face
    el_hatch = ax.contourf(X, Y, Z_el_mask, levels=[0.5, 1.5], colors='none', hatches=['//'])
    for collection in el_hatch.collections:
        collection.set_facecolor('none')   # Forces absolutely no fill color
        collection.set_edgecolor('#111111') # Controls line hatch color explicitly
        collection.set_alpha(0.35)          # Softens the harshness of the lines over the color data
    
    # Right-hand Y-Axis mapping: Distance translation scale
    ax_right = ax.twinx()
    ax_right.set_yscale('log')
    ax_right.set_ylim(semi_major_axis.max() / au, semi_major_axis.min() / au) 
    ax_right.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
    ax_right.yaxis.set_minor_formatter(ticker.NullFormatter())
    
    # BOUNDARY LOGIC & HATCHING LAYER: Dynamic check between blow-off and RR regimes
    if Z_transonic.max() <= Z_transonic.min():
               
        # COMBINED MASK: Active only where Z_transonic is 0 AND it's NOT the Energy-Limited regime (Z_el_mask == 0)
        mask_RR = ((Z_transonic == 0) & (Z_el_mask == 0)).astype(float)

        if mask_RR.any():
            # Use contourf to target exclusively the un-hatched, pure RR escape region
            bo_hatch = ax.contourf(X, Y, mask_RR, levels=[0.5, 1.5], colors='none', hatches=['\\\\'])
            for collection in bo_hatch.collections:
                collection.set_facecolor('none')   # No background fill color
                collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
                collection.set_alpha(0.40)          # Control line intensity
                collection.set_zorder(2)
    else:
        # 1. Generate an isolated, unfilled contour layer for the blow-off area using opposite hatching
        bo_hatch = ax.contourf(X, Y, Z_transonic, levels=[-0.5, 0.5], colors='none', hatches=['\\\\'])
        for collection in bo_hatch.collections:
            collection.set_facecolor('none')   # No background fill color
            collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
            collection.set_alpha(0.40)          # Control line intensity
            collection.set_zorder(2)
            
        # 2. Outline the boundary with the cyan dashed line to isolate the domain precisely
        ax.contour(X, Y, Z_transonic, levels=[0.5], colors='cyan', linestyles='--', linewidths=2.0, zorder=10)
        
    # --- SUB-NEPTUNE / SUPER-EARTH BOUNDARY ---
    mass_boundary = 7.6
    ax.axvline(x=mass_boundary, color='#333333', linestyle='-.', linewidth=1.5, alpha=0.9, zorder=4)
    
    # RESTORED: Original contrast-enhanced text labels (Bold layout restored)
    if i == 0:
        text_kwargs = dict(
            color='white', fontsize=15, rotation=90, va='top', weight='bold',
            bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=3)
        )
        y_text_pos = fluxes_plot.max() * 0.015
        
        ax.text(mass_boundary - 0.2, y_text_pos, 'Super-Earths', ha='right', **text_kwargs)
        ax.text(mass_boundary + 0.2, y_text_pos, 'Sub-Neptunes', ha='left', **text_kwargs)
    
    # Subplot baseline typography & visual grid formatting
    ax.set_title(case['label'], fontsize=18, pad=14)
    ax.grid(True, alpha=0.3, linestyle=':', color='#777777', zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=16, length=8, width=1.5)
    ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)
    ax.xaxis.get_major_ticks()[0].label1.set_visible(False)
    
    # Structural handling of isolated right-edge axes text labels
    is_right_edge = (i % num_cols) == (num_cols - 1) or (i == num_plots - 1)
    if is_right_edge:
        ax_right.tick_params(axis='y', which='major', labelsize=16, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', length=5, width=1.0)
    else:
        ax_right.tick_params(axis='y', which='major', labelleft=False, labelright=False, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', labelleft=False, labelright=False, length=5, width=1.0)

# Prune residual subplots inside empty indices
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])


# ====================================================================
# 3. GLOBAL CAPTIONS, COLORBARS & EXPORT GEOMETRY
# ====================================================================
# DYNAMIC CHECK: Determine if any panel triggered the global fallback state
has_blowoff_fallback = any(data['Z_transonic'].max() <= data['Z_transonic'].min() for data in data_to_plot)

# Build legend handles conditionally
handles = [
    patches.Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Radiation-Recombination Limited'),
    patches.Patch(facecolor='none', edgecolor='black', hatch='//', label='Energy-Limited Regime (Hatched)'),
]

if has_blowoff_fallback:
    handles.append(patches.Patch(facecolor='none', edgecolor='cyan', hatch='\\\\', alpha=0.6, label='RR Wind State: Blow-off'))

# Always append the boundary line last
handles.append(Line2D([0], [0], color='#444444', linewidth=1.5, linestyle='-.', label=r'Planetary Type Boundary ($7.6\ M_\oplus$)'))

# Grid configuration: 2 columns if blow-off is active (making it 2x2), otherwise single-row or 3-col layout
num_legend_cols = 2 if has_blowoff_fallback else 3

# Render legend with enhanced font size (18) and adjusted lower anchor
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.46, 0.025), ncol=num_legend_cols, 
           fontsize=19, frameon=True, facecolor='white', edgecolor='#d0d0d0', handletextpad=0.6, columnspacing=1.8)

# Pulled bottom margin slightly up to accommodate the larger 2x2 legend text footprint without overlapping labels
plt.subplots_adjust(left=0.10, right=0.77, hspace=0.30, wspace=0.22, bottom=0.16, top=0.89)

# Dynamic subscripting layout logic for target gas chemistry equations
fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in target_species])

# Main title
fig.suptitle(f'Dominant Hydrodynamic Escape Mechanisms for $\mathrm{{{fmt_species}}}$-Dominated Atmospheres\nAcross FGK Stars & Stellar Ages', 
             fontsize=24, y=0.97)

# Labels (Shifted supxlabel lower to clear space for the taller 2x2 layout)
fig.supxlabel(r'Planetary Mass [$M_\oplus$]', fontsize=20, y=0.09)
fig.supylabel(r'Received XUV Flux [$F_{\mathrm{XUV},\oplus}$]', fontsize=20, x=0.03)

# Adjust text position
fig.text(0.82, 0.52, r'Semi-Major Axis Distance [au]', fontsize=20, rotation=-90, va='center', ha='center')

# Reposition colorbar
cbar_ax = fig.add_axes([0.87, 0.16, 0.016, 0.73])
fig.colorbar(pcm, cax=cbar_ax)
cbar_ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=20, labelpad=20, rotation=270, va='bottom')
cbar_ax.tick_params(axis='y', which='major', labelsize=16, length=7, width=1.5)

# Save with tight margins
plt.savefig(f'Keepers/{target_species}_massfluxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ====================================================================
# 1. SETUP TARGET CONFIGURATION (Optimized Layout & Sizing)
# ====================================================================
target_species = 'H2O'  # e.g., 'H2O', 'CO2', 'H2', 'N2'
data_to_plot = computed_data_all_species[target_species]

num_plots = len(data_to_plot)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

# Figure setup: Slightly wider to prevent label clipping on the right
fig, axes = plt.subplots(num_rows, num_cols, figsize=(17.5, 5.8 * num_rows), sharex=True, sharey=False)
axes = np.atleast_1d(axes).flatten()

# ====================================================================
# 2. ITERATE AND RENDER PLOTS
# ====================================================================
for i, data in enumerate(data_to_plot):
    ax = axes[i]
    
    # Direct dictionary variable unpacking
    case, X, Y, Z_mdot, fluxes_plot = data['case'], data['X'], data['Y'], data['Z_mdot'], data['fluxes_plot']
    Z_el_mask = (~data['Z_mask'].astype(bool)).astype(float)
    Z_transonic = data['Z_transonic']
    
    # Base configuration: Flux coordinate system
    ax.set_yscale('log')
    ax.set_ylim(fluxes_plot.min(), fluxes_plot.max())
    
    # Primary continuous mass loss field mapping (FIXED FOR PDF SEAMLESS RENDERING)
    pcm = ax.pcolormesh(X, Y, Z_mdot, 
                        norm=colors.LogNorm(vmin=1e-2, vmax=1e11), 
                        cmap='plasma', 
                        shading='auto', 
                        edgecolors='face', # Closes the vector seams
                        linewidth=0)       # Ensures no artificial borders are drawn

    # FIXED: True transparent hatching layer using collection overrides instead of contourf
    el_contour = ax.contour(X, Y, Z_el_mask, levels=[0.5], colors='white', linewidths=2.5)
    
    # Generate an isolated hatching polygon path that is completely transparent on its face
    el_hatch = ax.contourf(X, Y, Z_el_mask, levels=[0.5, 1.5], colors='none', hatches=['//'])
    for collection in el_hatch.collections:
        collection.set_facecolor('none')   # Forces absolutely no fill color
        collection.set_edgecolor('#111111') # Controls line hatch color explicitly
        collection.set_alpha(0.35)          # Softens the harshness of the lines over the color data
    
    # Right-hand Y-Axis mapping: Distance translation scale
    ax_right = ax.twinx()
    ax_right.set_yscale('log')
    ax_right.set_ylim(semi_major_axis.max() / au, semi_major_axis.min() / au) 
    ax_right.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
    ax_right.yaxis.set_minor_formatter(ticker.NullFormatter())
    
    # BOUNDARY LOGIC & HATCHING LAYER: Dynamic check between blow-off and RR regimes
    if Z_transonic.max() <= Z_transonic.min():
               
        # COMBINED MASK: Active only where Z_transonic is 0 AND it's NOT the Energy-Limited regime (Z_el_mask == 0)
        mask_RR = ((Z_transonic == 0) & (Z_el_mask == 0)).astype(float)

        if mask_RR.any():
            # Use contourf to target exclusively the un-hatched, pure RR escape region
            bo_hatch = ax.contourf(X, Y, mask_RR, levels=[0.5, 1.5], colors='none', hatches=['\\\\'])
            for collection in bo_hatch.collections:
                collection.set_facecolor('none')   # No background fill color
                collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
                collection.set_alpha(0.40)          # Control line intensity
                collection.set_zorder(2)
    else:
        # 1. Generate an isolated, unfilled contour layer for the blow-off area using opposite hatching
        bo_hatch = ax.contourf(X, Y, Z_transonic, levels=[-0.5, 0.5], colors='none', hatches=['\\\\'])
        for collection in bo_hatch.collections:
            collection.set_facecolor('none')   # No background fill color
            collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
            collection.set_alpha(0.40)          # Control line intensity
            collection.set_zorder(2)
            
        # 2. Outline the boundary with the cyan dashed line to isolate the domain precisely
        ax.contour(X, Y, Z_transonic, levels=[0.5], colors='cyan', linestyles='--', linewidths=2.0, zorder=10)
        
    # --- SUB-NEPTUNE / SUPER-EARTH BOUNDARY ---
    mass_boundary = 7.6
    ax.axvline(x=mass_boundary, color='#333333', linestyle='-.', linewidth=1.5, alpha=0.9, zorder=4)
    
    # RESTORED: Original contrast-enhanced text labels (Bold layout restored)
    if i == 0:
        text_kwargs = dict(
            color='white', fontsize=15, rotation=90, va='top', weight='bold',
            bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=3)
        )
        y_text_pos = fluxes_plot.max() * 0.015
        
        ax.text(mass_boundary - 0.2, y_text_pos, 'Super-Earths', ha='right', **text_kwargs)
        ax.text(mass_boundary + 0.2, y_text_pos, 'Sub-Neptunes', ha='left', **text_kwargs)
    
    # Subplot baseline typography & visual grid formatting
    ax.set_title(case['label'], fontsize=18, pad=14)
    ax.grid(True, alpha=0.3, linestyle=':', color='#777777', zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=16, length=8, width=1.5)
    ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)
    ax.xaxis.get_major_ticks()[0].label1.set_visible(False)
    
    # Structural handling of isolated right-edge axes text labels
    is_right_edge = (i % num_cols) == (num_cols - 1) or (i == num_plots - 1)
    if is_right_edge:
        ax_right.tick_params(axis='y', which='major', labelsize=16, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', length=5, width=1.0)
    else:
        ax_right.tick_params(axis='y', which='major', labelleft=False, labelright=False, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', labelleft=False, labelright=False, length=5, width=1.0)

# Prune residual subplots inside empty indices
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])


# ====================================================================
# 3. GLOBAL CAPTIONS, COLORBARS & EXPORT GEOMETRY
# ====================================================================
# DYNAMIC CHECKS: Check conditions across all datasets to decide legend elements
has_blowoff_fallback = any(data['Z_transonic'].max() <= data['Z_transonic'].min() for data in data_to_plot)
has_boundary         = any(data['Z_transonic'].max() >  data['Z_transonic'].min() for data in data_to_plot)

# 1. Initialize base legend items that always exist
handles = [
    patches.Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Radiation-Recombination Limited'),
    patches.Patch(facecolor='none', edgecolor='black', hatch='//', label='Energy-Limited Regime (Hatched)'),
]

# 2. Conditionally append the Cyan Blow-off Hatching
if has_blowoff_fallback:
    handles.append(patches.Patch(facecolor='none', edgecolor='cyan', hatch='\\\\', alpha=0.6, label='RR Wind State: Blow-off'))

# 3. Conditionally append the Cyan Boundary Line (New!)
if has_boundary:
    handles.append(Line2D([0], [0], color='cyan', linewidth=2.0, linestyle='--', label='Blow-off Regime Boundary'))

# 4. Always append the gray planetary type boundary line last
handles.append(Line2D([0], [0], color='#444444', linewidth=1.5, linestyle='-.', label=r'Planetary Type Boundary ($7.6\ M_\oplus$)'))

# Dynamically set columns based on the total number of items to keep it clean (e.g., 2 columns for a neat 2x2 or 2x3 layout)
num_legend_cols = 2 if len(handles) > 3 else 3

# Render legend with enhanced font size (18) and adjusted lower anchor
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.46, 0.03), ncol=num_legend_cols, 
           fontsize=18, frameon=True, facecolor='white', edgecolor='#d0d0d0', handletextpad=0.6, columnspacing=1.8)

# Pulled bottom margin up slightly to accommodate the larger multi-line legend footprint without overlapping labels
plt.subplots_adjust(left=0.10, right=0.77, hspace=0.30, wspace=0.22, bottom=0.15, top=0.89)

# Dynamic subscripting layout logic for target gas chemistry equations
fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in target_species])

# Main title
fig.suptitle(f'Dominant Hydrodynamic Escape Mechanisms for $\mathrm{{{fmt_species}}}$-Dominated Atmospheres\nAcross FGK Stars & Stellar Ages', 
             fontsize=24, y=0.97)

# Labels (Shifted supxlabel lower to clear space for the taller layout)
fig.supxlabel(r'Planetary Mass [$M_\oplus$]', fontsize=20, y=0.1)
fig.supylabel(r'Received XUV Flux [$F_{\mathrm{XUV},\oplus}$]', fontsize=20, x=0.02)

# Adjust text position
fig.text(0.82, 0.52, r'Semi-Major Axis Distance [au]', fontsize=20, rotation=-90, va='center', ha='center')

# Reposition colorbar
cbar_ax = fig.add_axes([0.87, 0.17, 0.016, 0.72])
fig.colorbar(pcm, cax=cbar_ax)
cbar_ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=20, labelpad=20, rotation=270, va='bottom')
cbar_ax.tick_params(axis='y', which='major', labelsize=16, length=7, width=1.5)

# Save with tight margins
plt.savefig(f'Keepers/{target_species}_massfluxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ====================================================================
# 1. SETUP TARGET CONFIGURATION (Optimized Layout & Sizing)
# ====================================================================
target_species = 'CO2'  # e.g., 'H2O', 'CO2', 'H2', 'N2'
data_to_plot = computed_data_all_species[target_species]

num_plots = len(data_to_plot)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

# Figure setup: Slightly wider to prevent label clipping on the right
fig, axes = plt.subplots(num_rows, num_cols, figsize=(17.5, 5.8 * num_rows), sharex=True, sharey=False)
axes = np.atleast_1d(axes).flatten()

# ====================================================================
# 2. ITERATE AND RENDER PLOTS
# ====================================================================
for i, data in enumerate(data_to_plot):
    ax = axes[i]
    
    # Direct dictionary variable unpacking
    case, X, Y, Z_mdot, fluxes_plot = data['case'], data['X'], data['Y'], data['Z_mdot'], data['fluxes_plot']
    Z_el_mask = (~data['Z_mask'].astype(bool)).astype(float)
    Z_transonic = data['Z_transonic']
    
    # Base configuration: Flux coordinate system
    ax.set_yscale('log')
    ax.set_ylim(fluxes_plot.min(), fluxes_plot.max())
    
    # Primary continuous mass loss field mapping (FIXED FOR PDF SEAMLESS RENDERING)
    pcm = ax.pcolormesh(X, Y, Z_mdot, 
                        norm=colors.LogNorm(vmin=1e-2, vmax=1e11), 
                        cmap='plasma', 
                        shading='auto', 
                        edgecolors='face', # Closes the vector seams
                        linewidth=0)       # Ensures no artificial borders are drawn

    # FIXED: True transparent hatching layer using collection overrides instead of contourf
    el_contour = ax.contour(X, Y, Z_el_mask, levels=[0.5], colors='white', linewidths=2.5)
    
    # Generate an isolated hatching polygon path that is completely transparent on its face
    el_hatch = ax.contourf(X, Y, Z_el_mask, levels=[0.5, 1.5], colors='none', hatches=['//'])
    for collection in el_hatch.collections:
        collection.set_facecolor('none')   # Forces absolutely no fill color
        collection.set_edgecolor('#111111') # Controls line hatch color explicitly
        collection.set_alpha(0.35)          # Softens the harshness of the lines over the color data
    
    # Right-hand Y-Axis mapping: Distance translation scale
    ax_right = ax.twinx()
    ax_right.set_yscale('log')
    ax_right.set_ylim(semi_major_axis.max() / au, semi_major_axis.min() / au) 
    ax_right.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
    ax_right.yaxis.set_minor_formatter(ticker.NullFormatter())
    
    # BOUNDARY LOGIC & HATCHING LAYER: Dynamic check between blow-off and RR regimes
    if Z_transonic.max() <= Z_transonic.min():
        # COMBINED MASK: Active only where Z_transonic is 0 AND it's NOT the Energy-Limited regime (Z_el_mask == 0)
        mask_RR = ((Z_transonic == 0) & (Z_el_mask == 0)).astype(float)

        if mask_RR.any():
            # Use contourf to target exclusively the un-hatched, pure RR escape region
            bo_hatch = ax.contourf(X, Y, mask_RR, levels=[0.5, 1.5], colors='none', hatches=['\\\\'])
            for collection in bo_hatch.collections:
                collection.set_facecolor('none')   # No background fill color
                collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
                collection.set_alpha(0.40)          # Control line intensity
                collection.set_zorder(2)
    else:
        # 1. Generate an isolated, unfilled contour layer for the blow-off area using opposite hatching
        bo_hatch = ax.contourf(X, Y, Z_transonic, levels=[-0.5, 0.5], colors='none', hatches=['\\\\'])
        for collection in bo_hatch.collections:
            collection.set_facecolor('none')   # No background fill color
            collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
            collection.set_alpha(0.40)          # Control line intensity
            collection.set_zorder(2)
            
        # 2. Outline the boundary with the cyan dashed line to isolate the domain precisely
        ax.contour(X, Y, Z_transonic, levels=[0.5], colors='cyan', linestyles='--', linewidths=2.0, zorder=10)
        
    # --- SUB-NEPTUNE / SUPER-EARTH BOUNDARY ---
    mass_boundary = 7.6
    ax.axvline(x=mass_boundary, color='#333333', linestyle='-.', linewidth=1.5, alpha=0.9, zorder=4)
    
    # RESTORED: Original contrast-enhanced text labels (Bold layout restored)
    if i == 0:
        text_kwargs = dict(
            color='white', fontsize=15, rotation=90, va='top', weight='bold',
            bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=3)
        )
        y_text_pos = fluxes_plot.max() * 0.015
        
        ax.text(mass_boundary - 0.2, y_text_pos, 'Super-Earths', ha='right', **text_kwargs)
        ax.text(mass_boundary + 0.2, y_text_pos, 'Sub-Neptunes', ha='left', **text_kwargs)
    
    # Subplot baseline typography & visual grid formatting
    ax.set_title(case['label'], fontsize=18, pad=14)
    ax.grid(True, alpha=0.3, linestyle=':', color='#777777', zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=16, length=8, width=1.5)
    ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)
    ax.xaxis.get_major_ticks()[0].label1.set_visible(False)
    
    # Structural handling of isolated right-edge axes text labels
    is_right_edge = (i % num_cols) == (num_cols - 1) or (i == num_plots - 1)
    if is_right_edge:
        ax_right.tick_params(axis='y', which='major', labelsize=16, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', length=5, width=1.0)
    else:
        ax_right.tick_params(axis='y', which='major', labelleft=False, labelright=False, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', labelleft=False, labelright=False, length=5, width=1.0)

# Prune residual subplots inside empty indices
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])


# ====================================================================
# 3. GLOBAL CAPTIONS, COLORBARS & EXPORT GEOMETRY
# ====================================================================
# FIXED: Safe array checks verify if 0 (Blow-off regime indicator) is present anywhere in your datasets
has_blowoff_fallback = any((data['Z_transonic'].max() <= data['Z_transonic'].min()) and (0 in data['Z_transonic']) for data in data_to_plot)
has_boundary         = any(data['Z_transonic'].max() > data['Z_transonic'].min() for data in data_to_plot)

# 1. Initialize base legend items that always exist
handles = [
    patches.Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Radiation-Recombination Limited'),
    patches.Patch(facecolor='none', edgecolor='black', hatch='//', label='Energy-Limited Regime (Hatched)'),
]

# 2. Conditionally append the Cyan Blow-off Hatching
if has_blowoff_fallback:
    handles.append(patches.Patch(facecolor='none', edgecolor='cyan', hatch='\\\\', alpha=0.6, label='RR Wind State: Blow-off'))

# 3. Conditionally append the Cyan Boundary Line
if has_boundary:
    handles.append(Line2D([0], [0], color='cyan', linewidth=2.0, linestyle='--', label='Blow-off Regime Boundary'))

# 4. Always append the gray planetary type boundary line last
handles.append(Line2D([0], [0], color='#444444', linewidth=1.5, linestyle='-.', label=r'Planetary Type Boundary ($7.6\ M_\oplus$)'))

# Grid configuration: 2 columns if extra items exist (making a 2x2 or 2x3 layout), otherwise cleanly spreads across 3 columns
num_legend_cols = 2 if len(handles) > 3 else 3

# Render legend with enhanced font size (18) and adjusted lower anchor
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.46, 0.02), ncol=num_legend_cols, 
           fontsize=18, frameon=True, facecolor='white', edgecolor='#d0d0d0', handletextpad=0.6, columnspacing=1.8)

# Rebalanced bottom layout margins to perfectly separate the legend from data labels
plt.subplots_adjust(left=0.10, right=0.77, hspace=0.30, wspace=0.22, bottom=0.16, top=0.89)

# Dynamic subscripting layout logic for target gas chemistry equations
fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in target_species])

# Main title
fig.suptitle(f'Dominant Hydrodynamic Escape Mechanisms for $\mathrm{{{fmt_species}}}$-Dominated Atmospheres\nAcross FGK Stars & Stellar Ages', 
             fontsize=24, y=0.97)

# Labels
fig.supxlabel(r'Planetary Mass [$M_\oplus$]', fontsize=20, y=0.09)
fig.supylabel(r'Received XUV Flux [$F_{\mathrm{XUV},\oplus}$]', fontsize=20, x=0.02)

# Adjust text position
fig.text(0.82, 0.52, r'Semi-Major Axis Distance [au]', fontsize=20, rotation=-90, va='center', ha='center')

# Reposition colorbar
cbar_ax = fig.add_axes([0.87, 0.16, 0.016, 0.73])
fig.colorbar(pcm, cax=cbar_ax)
cbar_ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=20, labelpad=20, rotation=270, va='bottom')
cbar_ax.tick_params(axis='y', which='major', labelsize=16, length=7, width=1.5)

# Save with tight margins
plt.savefig(f'Keepers/{target_species}_massfluxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ====================================================================
# 1. SETUP TARGET CONFIGURATION (Optimized Layout & Sizing)
# ====================================================================
target_species = 'N2'  # e.g., 'H2O', 'CO2', 'H2', 'N2'
data_to_plot = computed_data_all_species[target_species]

num_plots = len(data_to_plot)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

# Figure setup: Slightly wider to prevent label clipping on the right
fig, axes = plt.subplots(num_rows, num_cols, figsize=(17.5, 5.8 * num_rows), sharex=True, sharey=False)
axes = np.atleast_1d(axes).flatten()

# ====================================================================
# 2. ITERATE AND RENDER PLOTS
# ====================================================================
for i, data in enumerate(data_to_plot):
    ax = axes[i]
    
    # Direct dictionary variable unpacking
    case, X, Y, Z_mdot, fluxes_plot = data['case'], data['X'], data['Y'], data['Z_mdot'], data['fluxes_plot']
    Z_el_mask = (~data['Z_mask'].astype(bool)).astype(float)
    Z_transonic = data['Z_transonic']
    
    # Base configuration: Flux coordinate system
    ax.set_yscale('log')
    ax.set_ylim(fluxes_plot.min(), fluxes_plot.max())
    
    # Primary continuous mass loss field mapping (FIXED FOR PDF SEAMLESS RENDERING)
    pcm = ax.pcolormesh(X, Y, Z_mdot, 
                        norm=colors.LogNorm(vmin=1e-2, vmax=1e11), 
                        cmap='plasma', 
                        shading='auto', 
                        edgecolors='face', # Closes the vector seams
                        linewidth=0)       # Ensures no artificial borders are drawn

    # FIXED: True transparent hatching layer using collection overrides instead of contourf
    el_contour = ax.contour(X, Y, Z_el_mask, levels=[0.5], colors='white', linewidths=2.5)
    
    # Generate an isolated hatching polygon path that is completely transparent on its face
    el_hatch = ax.contourf(X, Y, Z_el_mask, levels=[0.5, 1.5], colors='none', hatches=['//'])
    for collection in el_hatch.collections:
        collection.set_facecolor('none')   # Forces absolutely no fill color
        collection.set_edgecolor('#111111') # Controls line hatch color explicitly
        collection.set_alpha(0.35)          # Softens the harshness of the lines over the color data
    
    # Right-hand Y-Axis mapping: Distance translation scale
    ax_right = ax.twinx()
    ax_right.set_yscale('log')
    ax_right.set_ylim(semi_major_axis.max() / au, semi_major_axis.min() / au) 
    ax_right.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
    ax_right.yaxis.set_minor_formatter(ticker.NullFormatter())
    
    # BOUNDARY LOGIC & HATCHING LAYER: Dynamic check between blow-off and RR regimes
    if Z_transonic.max() <= Z_transonic.min():
        # COMBINED MASK: Active only where Z_transonic is 0 AND it's NOT the Energy-Limited regime (Z_el_mask == 0)
        mask_RR = ((Z_transonic == 0) & (Z_el_mask == 0)).astype(float)

        if mask_RR.any():
            # Use contourf to target exclusively the un-hatched, pure RR escape region
            bo_hatch = ax.contourf(X, Y, mask_RR, levels=[0.5, 1.5], colors='none', hatches=['\\\\'])
            for collection in bo_hatch.collections:
                collection.set_facecolor('none')   # No background fill color
                collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
                collection.set_alpha(0.40)          # Control line intensity
                collection.set_zorder(2)
    else:
        # 1. Generate an isolated, unfilled contour layer for the blow-off area using opposite hatching
        bo_hatch = ax.contourf(X, Y, Z_transonic, levels=[-0.5, 0.5], colors='none', hatches=['\\\\'])
        for collection in bo_hatch.collections:
            collection.set_facecolor('none')   # No background fill color
            collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
            collection.set_alpha(0.40)          # Control line intensity
            collection.set_zorder(2)
            
        # 2. Outline the boundary with the cyan dashed line to isolate the domain precisely
        ax.contour(X, Y, Z_transonic, levels=[0.5], colors='cyan', linestyles='--', linewidths=2.0, zorder=10)
        
    # --- SUB-NEPTUNE / SUPER-EARTH BOUNDARY ---
    mass_boundary = 7.6
    ax.axvline(x=mass_boundary, color='#333333', linestyle='-.', linewidth=1.5, alpha=0.9, zorder=4)
    
    # RESTORED: Original contrast-enhanced text labels (Bold layout restored)
    if i == 0:
        text_kwargs = dict(
            color='white', fontsize=15, rotation=90, va='top', weight='bold',
            bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=3)
        )
        y_text_pos = fluxes_plot.max() * 0.015
        
        ax.text(mass_boundary - 0.2, y_text_pos, 'Super-Earths', ha='right', **text_kwargs)
        ax.text(mass_boundary + 0.2, y_text_pos, 'Sub-Neptunes', ha='left', **text_kwargs)
    
    # Subplot baseline typography & visual grid formatting
    ax.set_title(case['label'], fontsize=18, pad=14)
    ax.grid(True, alpha=0.3, linestyle=':', color='#777777', zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=16, length=8, width=1.5)
    ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)
    ax.xaxis.get_major_ticks()[0].label1.set_visible(False)
    
    # Structural handling of isolated right-edge axes text labels
    is_right_edge = (i % num_cols) == (num_cols - 1) or (i == num_plots - 1)
    if is_right_edge:
        ax_right.tick_params(axis='y', which='major', labelsize=16, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', length=5, width=1.0)
    else:
        ax_right.tick_params(axis='y', which='major', labelleft=False, labelright=False, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', labelleft=False, labelright=False, length=5, width=1.0)

# Prune residual subplots inside empty indices
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])


# ====================================================================
# 3. GLOBAL CAPTIONS, COLORBARS & EXPORT GEOMETRY
# ====================================================================
# FIXED: Safe array checks verify if 0 (Blow-off regime indicator) is present anywhere in your datasets
has_blowoff_fallback = any((data['Z_transonic'].max() <= data['Z_transonic'].min()) and (0 in data['Z_transonic']) for data in data_to_plot)
has_boundary         = any(data['Z_transonic'].max() > data['Z_transonic'].min() for data in data_to_plot)

# 1. Initialize base legend items that always exist
handles = [
    patches.Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Radiation-Recombination Limited'),
    patches.Patch(facecolor='none', edgecolor='black', hatch='//', label='Energy-Limited Regime (Hatched)'),
]

# 2. Conditionally append the Cyan Blow-off Hatching
if has_blowoff_fallback:
    handles.append(patches.Patch(facecolor='none', edgecolor='cyan', hatch='\\\\', alpha=0.6, label='RR Wind State: Blow-off'))

# 3. Conditionally append the Cyan Boundary Line
if has_boundary:
    handles.append(Line2D([0], [0], color='cyan', linewidth=2.0, linestyle='--', label='Blow-off Regime Boundary'))

# 4. Always append the gray planetary type boundary line last
handles.append(Line2D([0], [0], color='#444444', linewidth=1.5, linestyle='-.', label=r'Planetary Type Boundary ($7.6\ M_\oplus$)'))

# Grid configuration: 2 columns if extra items exist (making a 2x2 or 2x3 layout), otherwise cleanly spreads across 3 columns
num_legend_cols = 2 if len(handles) > 3 else 3

# Render legend with enhanced font size (18) and adjusted lower anchor
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.46, 0.02), ncol=num_legend_cols, 
           fontsize=18, frameon=True, facecolor='white', edgecolor='#d0d0d0', handletextpad=0.6, columnspacing=1.8)

# Rebalanced bottom layout margins to perfectly separate the legend from data labels
plt.subplots_adjust(left=0.10, right=0.77, hspace=0.30, wspace=0.22, bottom=0.16, top=0.89)

# Dynamic subscripting layout logic for target gas chemistry equations
fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in target_species])

# Main title
fig.suptitle(f'Dominant Hydrodynamic Escape Mechanisms for $\mathrm{{{fmt_species}}}$-Dominated Atmospheres\nAcross FGK Stars & Stellar Ages', 
             fontsize=24, y=0.97)

# Labels
fig.supxlabel(r'Planetary Mass [$M_\oplus$]', fontsize=20, y=0.09)
fig.supylabel(r'Received XUV Flux [$F_{\mathrm{XUV},\oplus}$]', fontsize=20, x=0.02)

# Adjust text position
fig.text(0.82, 0.52, r'Semi-Major Axis Distance [au]', fontsize=20, rotation=-90, va='center', ha='center')

# Reposition colorbar
cbar_ax = fig.add_axes([0.87, 0.16, 0.016, 0.73])
fig.colorbar(pcm, cax=cbar_ax)
cbar_ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=20, labelpad=20, rotation=270, va='bottom')
cbar_ax.tick_params(axis='y', which='major', labelsize=16, length=7, width=1.5)

# Save with tight margins
plt.savefig(f'Keepers/{target_species}_massfluxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ====================================================================
# 1. SETUP TARGET CONFIGURATION (Fixed Case, Multiple Species)
# ====================================================================
# Define the order of compositions you want to showcase across the panels
all_species = ['H2', 'H2O', 'CO2', 'N2'] 

num_plots = len(all_species)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

# Figure setup optimized for A4 layout
fig, axes = plt.subplots(num_rows, num_cols, figsize=(16.5, 5.8 * num_rows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()

# Old G type star index
case_idx = 3 

# ====================================================================
# 2. ITERATE THROUGH COMPOSITIONS FOR THE OLD G-STAR CASE
# ====================================================================
for i, species in enumerate(all_species):
    ax = axes[i]
    
    # Extract the data specifically for this species and the chosen case
    species_data_list = computed_data_all_species[species]
    data = species_data_list[case_idx]
    
    # Direct dictionary variable unpacking
    case, X, Y, Z_mdot, fluxes_plot = data['case'], data['X'], data['Y'], data['Z_mdot'], data['fluxes_plot']
    Z_el_mask = (~data['Z_mask'].astype(bool)).astype(float)
    Z_transonic = data['Z_transonic']
    
    # Base configuration: Flux coordinate system
    ax.set_yscale('log')
    ax.set_ylim(fluxes_plot.min(), fluxes_plot.max())
    
    # Primary continuous mass loss field mapping
    pcm = ax.pcolormesh(X, Y, Z_mdot, 
                        norm=colors.LogNorm(vmin=1e-2, vmax=1e11), 
                        cmap='plasma', 
                        shading='auto',
                        edgecolors='face',
                        linewidth=0)

    # True transparent hatching layer using collection overrides instead of contourf
    el_contour = ax.contour(X, Y, Z_el_mask, levels=[0.5], colors='white', linewidths=2.5)
    
    # Generate an isolated hatching polygon path that is completely transparent on its face
    el_hatch = ax.contourf(X, Y, Z_el_mask, levels=[0.5, 1.5], colors='none', hatches=['//'])
    for collection in el_hatch.collections:
        collection.set_facecolor('none')   # Forces absolutely no fill color
        collection.set_edgecolor('#111111') # Controls line hatch color explicitly
        collection.set_alpha(0.35)          # Softens the harshness of the lines over the color data
    
    # Right-hand Y-Axis mapping: Distance translation scale
    ax_right = ax.twinx()
    ax_right.set_yscale('log')
    ax_right.set_ylim(semi_major_axis.max() / au, semi_major_axis.min() / au) 
    ax_right.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
    ax_right.yaxis.set_minor_formatter(ticker.NullFormatter())
    
    # BOUNDARY LOGIC & HATCHING LAYER: Dynamic check between blow-off and RR regimes
    if Z_transonic.max() <= Z_transonic.min():
        # COMBINED MASK: Active only where Z_transonic is 0 AND it's NOT the Energy-Limited regime (Z_el_mask == 0)
        mask_RR = ((Z_transonic == 0) & (Z_el_mask == 0)).astype(float)

        if mask_RR.any():
            # Use contourf to target exclusively the un-hatched, pure RR escape region
            bo_hatch = ax.contourf(X, Y, mask_RR, levels=[0.5, 1.5], colors='none', hatches=['\\\\'])
            for collection in bo_hatch.collections:
                collection.set_facecolor('none')   # No background fill color
                collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
                collection.set_alpha(0.40)          # Control line intensity
                collection.set_zorder(2)
    else:
        # 1. Generate an isolated, unfilled contour layer for the blow-off area using opposite hatching
        bo_hatch = ax.contourf(X, Y, Z_transonic, levels=[-0.5, 0.5], colors='none', hatches=['\\\\'])
        for collection in bo_hatch.collections:
            collection.set_facecolor('none')   # No background fill color
            collection.set_edgecolor('cyan')   # Explicitly color the hatch lines cyan
            collection.set_alpha(0.40)          # Control line intensity
            collection.set_zorder(2)
            
        # 2. Outline the boundary with the cyan dashed line to isolate the domain precisely
        ax.contour(X, Y, Z_transonic, levels=[0.5], colors='cyan', linestyles='--', linewidths=2.5, zorder=10)
        
    # --- SUB-NEPTUNE / SUPER-EARTH BOUNDARY ---
    mass_boundary = 7.6
    ax.axvline(x=mass_boundary, color='#333333', linestyle='-.', linewidth=1.5, alpha=0.9, zorder=4)
    
    # Original contrast-enhanced text labels (Bold layout retained)
    if i == 0:
        text_kwargs = dict(
            color='white', fontsize=14, rotation=90, va='top', weight='bold',
            bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', pad=3)
        )
        y_text_pos = fluxes_plot.max() * 0.025
        
        ax.text(mass_boundary - 0.2, y_text_pos, 'Super-Earths', ha='right', **text_kwargs)
        ax.text(mass_boundary + 0.2, y_text_pos, 'Sub-Neptunes', ha='left', **text_kwargs)
    
    # Subplot baseline typography & visual grid formatting (Title shows the chemistry type instead)
    fmt_species = "".join([f'_{char}' if char.isdigit() else char for char in species])
    ax.set_title(f'$\mathrm{{{fmt_species}}}$-Dominated', fontsize=18, pad=14)
    
    # NEW: Hard limit the panel frame at the sweep ceiling (16.2)
    ax.set_xlim(1.0, 16.2)
    
    
    ax.grid(True, alpha=0.3, linestyle=':', color='#777777', zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=16, length=8, width=1.5)
    ax.tick_params(axis='both', which='minor', labelsize=13, length=5, width=1.0)
    # Structural handling of isolated right-edge axes text labels
    is_right_edge = (i % num_cols) == (num_cols - 1) or (i == num_plots - 1)
    if is_right_edge:
        ax_right.tick_params(axis='y', which='major', labelsize=16, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', length=5, width=1.0)
    else:
        ax_right.tick_params(axis='y', which='major', labelleft=False, labelright=False, length=8, width=1.5)
        ax_right.tick_params(axis='y', which='minor', labelleft=False, labelright=False, length=5, width=1.0)

# Prune residual subplots inside empty indices
for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])

# ====================================================================
# 3. GLOBAL CAPTIONS, COLORBARS & EXPORT GEOMETRY
# ====================================================================
# DYNAMIC CHECKS: Scan current cross-sections for appropriate plotting legend criteria
has_blowoff_fallback = any(
    (computed_data_all_species[sp][case_idx]['Z_transonic'].max() <= computed_data_all_species[sp][case_idx]['Z_transonic'].min()) 
    and (0 in computed_data_all_species[sp][case_idx]['Z_transonic']) 
    for sp in all_species
)
has_boundary = any(
    computed_data_all_species[sp][case_idx]['Z_transonic'].max() > computed_data_all_species[sp][case_idx]['Z_transonic'].min() 
    for sp in all_species
)

# 1. Initialize base legend items that always exist
handles = [
    patches.Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Radiation-Recombination Limited'),
    patches.Patch(facecolor='none', edgecolor='black', hatch='//', label='Energy-Limited Regime (Hatched)'),
]

# 2. Conditionally append the Cyan Blow-off Hatching
if has_blowoff_fallback:
    handles.append(patches.Patch(facecolor='none', edgecolor='cyan', hatch='\\\\', alpha=0.6, label='RR Wind State: Blow-off'))

# 4. Always append the gray planetary type boundary line last
handles.append(Line2D([0], [0], color='#444444', linewidth=1.5, linestyle='-.', label=r'Planetary Type Boundary ($7.6\ M_\oplus$)'))

# Dynamically set grid column distribution
num_legend_cols = 2 if len(handles) > 3 else 3

# FIXED: Dropped bbox_to_anchor lower to 0.015 to isolate it cleanly below the X-axis label
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.45, 0.015), ncol=num_legend_cols, 
           fontsize=18, frameon=True, facecolor='white', edgecolor='#d0d0d0', handletextpad=0.6, columnspacing=1.8) 

# FIXED: Lifted bottom to 0.22 to clear massive breathing room for labels & legend matrix
plt.subplots_adjust(left=0.10, right=0.80, hspace=0.28, wspace=0.18, bottom=0.22, top=0.86)

# Global custom layout text mapping for target configuration overview
star_label = case['label'] 
fig.suptitle(f'Atmospheric Hydrodynamic Escape Regimes Across Chemical Compositions\nHost Star: {star_label}', 
             fontsize=24, y=0.98)

# FIXED: Adjusted y-positioning of the global X label so it floats perfectly halfway between plots and legend
fig.supxlabel(r'Planetary Mass [$M_\oplus$]', fontsize=20, y=0.13)
fig.supylabel(r'Received XUV Flux [$F_{\mathrm{XUV},\oplus}$]', fontsize=20, x=0.02)
fig.text(0.85, 0.54, r'Semi-Major Axis Distance [au]', fontsize=20, rotation=-90, va='center', ha='center')

# Colorbar adjusted to match the new panel bottom baseline
cbar_ax = fig.add_axes([0.90, 0.22, 0.018, 0.64])
fig.colorbar(pcm, cax=cbar_ax)
cbar_ax.set_ylabel(r'Mass-loss Rate $\dot{M}$ [$\mathrm{kg\ s}^{-1}$]', fontsize=20, labelpad=25, rotation=270, va='bottom')
cbar_ax.tick_params(axis='y', which='major', labelsize=16, length=7, width=1.5)

plt.savefig('Keepers/mature_G_star_compositions_massfluxplot.png', dpi=300, bbox_inches='tight')
plt.show()